# all-reduce-grad-sync — faded example 1: Complete the mean step of grad sync

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`. The last cell reports your progress on the `Distributed: all_reduce grad sync` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

After `all_reduce(grad, SUM)` each rank holds the SUM of all ranks' grads. To recover the global *mean* you must divide that buffer by `world_size`. This divide is the step that makes multi-GPU training equivalent to single-GPU training on the concatenated batch.

## Faded exercise 1

Two ranks have grads `[1, 1]` and `[3, 3]`. The mock `all_reduce` (SUM) has already been applied, so both buffers now hold `[4, 4]`. Complete the loop so each rank's grad becomes the *mean* across ranks.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
import numpy as np
import torch as t

def mock_all_reduce_sum(buffers):
    total = sum(b.clone() for b in buffers)
    for b in buffers:
        b.copy_(total)

t.manual_seed(0)
world_size = 2
grads = [t.tensor([1.0, 1.0]), t.tensor([3.0, 3.0])]
mock_all_reduce_sum(grads)   # both buffers now hold [4, 4]
for g in grads:
    g /= world_size


def _test():
    import torch as t
    expected = t.tensor([2.0, 2.0])
    assert t.allclose(grads[0], expected), grads[0]
    assert t.allclose(grads[1], expected), grads[1]
    assert t.allclose(grads[0], grads[1]), 'ranks must agree after sync'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import numpy as np
import torch as t

def mock_all_reduce_sum(buffers):
    total = sum(b.clone() for b in buffers)
    for b in buffers:
        b.copy_(total)

t.manual_seed(0)
world_size = 2
grads = [t.tensor([1.0, 1.0]), t.tensor([3.0, 3.0])]
mock_all_reduce_sum(grads)   # both buffers now hold [4, 4]
for g in grads:
    g /= world_size
```
</details>